In [ ]:
import warnings

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

# This is required to catch warnings when the multiprocessing module is used
import os

os.environ["PYTHONWARNINGS"] = "ignore"

import scanpy as sc
import pandas as pd
import numpy as np
import os, sys
import pertpy as pt
import matplotlib.pyplot as plt

In [ ]:
adata = pt.dt.kang_2018()
print(adata.obs['cell_type'].value_counts())


cell_type
CD4 T cells          11238
CD14+ Monocytes       5697
B cells               2651
NK cells              1716
CD8 T cells           1621
FCGR3A+ Monocytes     1089
Dendritic cells        529
Megakaryocytes         132
Name: count, dtype: int64


In [ ]:
adata.obs['ct'] = adata.obs['cell_type'].str.split(' ').str[0].str.lower()
print(adata.obs[['ct', 'cell_type']].value_counts())

ct              cell_type        
cd4             CD4 T cells          11238
cd14+           CD14+ Monocytes       5697
b               B cells               2651
nk              NK cells              1716
cd8             CD8 T cells           1621
fcgr3a+         FCGR3A+ Monocytes     1089
dendritic       Dendritic cells        529
megakaryocytes  Megakaryocytes         132
Name: count, dtype: int64


In [ ]:
sc.pp.filter_genes(adata, min_counts=3)
sc.pp.filter_cells(adata, min_genes=100)
adata.layers['raw'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(
    adata, n_top_genes=5000,
    subset=False, flavor='seurat',
)

In [ ]:
adata.obs["perturbation"] = adata.obs['label'].replace({"ctrl": "control", "stim": "stimulated"})

In [ ]:
adata.write_h5ad('preprocessed.h5ad')

## split dataset

In [ ]:
ad = sc.read(f'preprocessed.h5ad')
split_df = ad.obs.copy()a
ad.obs[['ct', 'cell_type']].value_counts()

ct              cell_type        
cd4             CD4 T cells          11238
cd14+           CD14+ Monocytes       5697
b               B cells               2651
nk              NK cells              1716
cd8             CD8 T cells           1621
fcgr3a+         FCGR3A+ Monocytes     1089
dendritic       Dendritic cells        529
megakaryocytes  Megakaryocytes          35
Name: count, dtype: int64

In [ ]:
np.random.seed(42)
split_df['split'] = np.random.choice(['train', 'val', 'test'], size=len(split_df), p=[0.6, 0.1, 0.3])
split_df = split_df.reset_index(names='cell')[['cell', 'split']]
split_df['subsplit'] = ad.obs['ct'].values
split_df.to_csv("split_random.csv", index=False)
split_df

,cell,split,subsplit
0,AAACATACATTTCC-1,train,cd14+
1,AAACATACCAGAAA-1,test,cd14+
2,AAACATACCATGCA-1,test,cd4
3,AAACATACCTCGCT-1,train,cd14+
4,AAACATACCTGGTA-1,train,dendritic
...,...,...,...
24571,TTTGCATGCCTGAA-2,train,cd4
24572,TTTGCATGCCTGTC-2,train,b
24573,TTTGCATGCTAAGC-2,train,cd4
24574,TTTGCATGGGACGA-2,val,cd4


In [ ]:
for ct in ad.obs['ct'].unique():
    print(ct)
    subsplit_df = split_df.query('subsplit == @ct').drop(columns='subsplit').copy()
    print(subsplit_df['split'].value_counts())
    subsplit_df.to_csv(f"split_random_{ct}.csv", index=False)

cd14+
split
train    3368
test     1742
val       587
Name: count, dtype: int64
cd4
split
train    6730
test     3381
val      1127
Name: count, dtype: int64
dendritic
split
train    335
test     135
val       59
Name: count, dtype: int64
nk
split
train    1044
test      501
val       171
Name: count, dtype: int64
cd8
split
train    974
test     490
val      157
Name: count, dtype: int64
b
split
train    1573
test      816
val       262
Name: count, dtype: int64
fcgr3a+
split
train    672
test     315
val      102
Name: count, dtype: int64
megakaryocytes
split
train    21
test      8
val       6
Name: count, dtype: int64


In [ ]:
np.savetxt(
    'ct_list.txt', 
    ad.obs['ct'].unique(), 
    fmt='%s'
)

In [ ]:
import pandas as pd
split_df = pd.read_csv('split_random.csv')

for ct in ['cd4', 'cd14+', 'b']:
    subsplit_df = split_df.copy()
    subsplit_df.loc[subsplit_df['subsplit'] != ct, 'split'] = 'test'
    subsplit_df.to_csv(f'split_random_{ct}_vr.csv', index=False)
    print(subsplit_df[['split', 'subsplit']].groupby(['subsplit', 'split']).size())
    

subsplit        split
b               test     2651
cd14+           test     5697
cd4             test     3381
                train    6730
                val      1127
cd8             test     1621
dendritic       test      529
fcgr3a+         test     1089
megakaryocytes  test       35
nk              test     1716
dtype: int64
subsplit        split
b               test      2651
cd14+           test      1742
                train     3368
                val        587
cd4             test     11238
cd8             test      1621
dendritic       test       529
fcgr3a+         test      1089
megakaryocytes  test        35
nk              test      1716
dtype: int64
subsplit        split
b               test       816
                train     1573
                val        262
cd14+           test      5697
cd4             test     11238
cd8             test      1621
dendritic       test       529
fcgr3a+         test      1089
megakaryocytes  test        35
nk              tes